Importing Libraries

In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt


Load Dataset

In [14]:
import tensorflow_datasets as tfds

dataset, info = tfds.load('cats_vs_dogs', with_info=True, as_supervised=True)
train_dataset = dataset['train']


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cats_vs_dogs/incomplete.G5EGGV_4.0.1/cats_vs_dogs-train.tfrecord*...:   0%…

Dataset cats_vs_dogs downloaded and prepared to /root/tensorflow_datasets/cats_vs_dogs/4.0.1. Subsequent calls will reuse this data.


Preprocessing & Sampling (250 Cats + 250 Dogs)

In [15]:
import random

cat_images = []
dog_images = []

for img, label in train_dataset.take(10000):  # read 10k samples
    if label.numpy() == 0 and len(cat_images) < 250:
        cat_images.append((img, label))
    elif label.numpy() == 1 and len(dog_images) < 250:
        dog_images.append((img, label))
    if len(cat_images) == 250 and len(dog_images) == 250:
        break

subset = cat_images + dog_images
random.shuffle(subset)


Resize and Normalize Images

In [16]:
import numpy as np
X = []
y = []

for img, label in subset:
    img = tf.image.resize(img, (128, 128)) / 255.0
    X.append(img.numpy())
    y.append(label.numpy())

X = np.array(X)
y = np.array(y)


Split into Train & Test

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Define CNN Model (with Dropout)

In [18]:
def build_cnn(dropout=True):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5 if dropout else 0.0),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


Train Models (With & Without Dropout)

In [19]:
cnn_drop = build_cnn(dropout=True)
cnn_no_drop = build_cnn(dropout=False)

hist_drop = cnn_drop.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), verbose=2)
hist_no_drop = cnn_no_drop.fit(X_train, y_train, epochs=10, validation_data=(X_test, y_test), verbose=2)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
13/13 - 18s - 1s/step - accuracy: 0.5025 - loss: 0.8181 - val_accuracy: 0.4500 - val_loss: 0.7238
Epoch 2/10
13/13 - 14s - 1s/step - accuracy: 0.4900 - loss: 0.6983 - val_accuracy: 0.4600 - val_loss: 0.6929
Epoch 3/10
13/13 - 14s - 1s/step - accuracy: 0.5450 - loss: 0.6901 - val_accuracy: 0.6500 - val_loss: 0.6827
Epoch 4/10
13/13 - 23s - 2s/step - accuracy: 0.6075 - loss: 0.6704 - val_accuracy: 0.5800 - val_loss: 0.6741
Epoch 5/10
13/13 - 14s - 1s/step - accuracy: 0.6475 - loss: 0.6452 - val_accuracy: 0.6100 - val_loss: 0.6611
Epoch 6/10
13/13 - 15s - 1s/step - accuracy: 0.6675 - loss: 0.5988 - val_accuracy: 0.5900 - val_loss: 0.6767
Epoch 7/10
13/13 - 20s - 2s/step - accuracy: 0.7350 - loss: 0.5219 - val_accuracy: 0.6100 - val_loss: 0.7579
Epoch 8/10
13/13 - 20s - 2s/step - accuracy: 0.7900 - loss: 0.4234 - val_accuracy: 0.6100 - val_loss: 0.7470
Epoch 9/10
13/13 - 20s - 2s/step - accuracy: 0.8325 - loss: 0.3685 - val_accuracy: 0.5900 - val_loss: 0.8594
Epoch 10/10
13/13 -

Evaluate Accuracy

In [20]:
acc_drop = cnn_drop.evaluate(X_test, y_test, verbose=0)[1]
acc_no_drop = cnn_no_drop.evaluate(X_test, y_test, verbose=0)[1]

print(f"Accuracy with Dropout: {acc_drop:.2f}")
print(f"Accuracy without Dropout: {acc_no_drop:.2f}")


Accuracy with Dropout: 0.60
Accuracy without Dropout: 0.61


No the accuracy does not remain same .
After reducing the dataset size to 500 images (250 cats and 250 dogs),
the model’s accuracy dropped significantly compared to the full dataset.
This shows that CNNs require more training data to generalize well.
Dropout can slightly improve generalization, but data quantity matters more.